# Fine-tuning de Spleeter (2 stems) con MUSDB18

Ajuste fino del modelo **Spleeter 2stems** (red U-Net de Deezer), que el
sistema de karaoke usa como modelo de respaldo para separar la voz del
acompañamiento. Se parte de los pesos preentrenados y se sigue entrenando
con el dataset **MUSDB18** (fragmentos de 7 s, licencia CC BY 4.0), el mismo
que está en `pretrained_models/spleeter/dataset/musdb18-7s`.

Pasos:
1. Preparar un entorno con Python 3.10 y Spleeter 2.4.0 (TensorFlow 2.9).
2. Descargar el dataset y convertirlo al formato de entrenamiento de
   Spleeter (WAV + CSV).
3. Medir el SDR del modelo preentrenado sobre el conjunto de prueba.
4. Entrenar (fine-tuning) partiendo de los pesos preentrenados.
5. Medir de nuevo el SDR y comparar.

En Colab: **Entorno de ejecución > Cambiar tipo de entorno > GPU**.

## 1. Entorno

Spleeter 2.4.0 necesita Python 3.10 y TensorFlow 2.9, más antiguos que los
de Colab, así que se instala en un entorno aparte con micromamba (con CUDA
11.2 y cuDNN 8.1 para usar la GPU).

In [ ]:
!nvidia-smi -L
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
!./bin/micromamba create -y -q -p /content/entorno_spleeter -c conda-forge python=3.10 cudatoolkit=11.2 cudnn=8.1.0
!/content/entorno_spleeter/bin/pip install -q spleeter==2.4.0

In [ ]:
import hashlib
import json
import os
import re
import subprocess
import tarfile
import time
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.io import wavfile

# Python del entorno de Spleeter
PY = os.getenv("PY_SPLEETER", "/content/entorno_spleeter/bin/python")
ENTORNO = dict(os.environ)
if Path("/content/entorno_spleeter/lib").is_dir():
    ENTORNO["LD_LIBRARY_PATH"] = "/content/entorno_spleeter/lib"


def spleeter(*argumentos, entorno=None):
    return subprocess.run(
        [PY, "-m", "spleeter", *argumentos],
        env=entorno or ENTORNO,
        check=True,
        capture_output=True,
        text=True,
    )


print(subprocess.run([PY, "--version"], capture_output=True, text=True).stdout)

## 2. Configuración del entrenamiento

In [ ]:
CONFIG = {
    "modelo_base": "spleeter:2stems",
    "dataset": "MUSDB18 (fragmentos de 7 s)",
    "pasos": int(os.getenv("PASOS", "3000")),
    "lote": int(os.getenv("LOTE", "4")),
    "tasa_aprendizaje": 1e-5,
    # Los fragmentos duran 6,8 s: T = 256 cuadros de espectrograma (5,9 s)
    # en lugar de 512, que exige al menos 11,9 s
    "T": 256,
    "duracion_fragmento": 6.5,
    "semilla": 42,
    # 0 = todas las pistas del conjunto
    "pistas_entrenamiento": int(os.getenv("PISTAS_ENTRENAMIENTO", "0")),
    "pistas_prueba": int(os.getenv("PISTAS_PRUEBA", "0")),
}
TRABAJO = Path(os.getenv("DIR_TRABAJO", ".")).resolve()
DIR_DATASET = Path(os.getenv("DIR_DATASET", TRABAJO / "musdb18-7s"))
DIR_WAV = TRABAJO / "wav"
DIR_PREENTRENADO = TRABAJO / "preentrenado"
DIR_SALIDA = TRABAJO / "modelo_afinado"
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

## 3. Dataset MUSDB18

Se descarga de Zenodo (registro 3270814) y se comprueba el MD5. Es el mismo
archivo guardado en el proyecto; también puede subirse desde Google Drive y
apuntar `DIR_DATASET` a esa carpeta.

In [ ]:
URL = (
    "https://zenodo.org/records/3270814/files/"
    "MUSDB18-7-STEMS.zip?download=1"
)
MD5 = "dc2bdfac5b46ce742bd60009866758a5"

if not (DIR_DATASET / "train").is_dir():
    zip_dataset = TRABAJO / "MUSDB18-7-STEMS.zip"
    if not zip_dataset.exists():
        print("Descargando el dataset...")
        urllib.request.urlretrieve(URL, zip_dataset)
    md5 = hashlib.md5(zip_dataset.read_bytes()).hexdigest()
    assert md5 == MD5, f"Checksum incorrecto: {md5}"
    with zipfile.ZipFile(zip_dataset) as z:
        z.extractall(DIR_DATASET)

### Formato de Spleeter

Cada `.stem.mp4` tiene 5 pistas: mezcla, batería, bajo, otros y voz. Para
2 stems se generan `mix.wav`, `vocals.wav` y `accompaniment.wav` (suma de
batería, bajo y otros), y un CSV por conjunto con las rutas y la duración.

In [ ]:
def preparar(subconjunto, limite):
    rutas = sorted((DIR_DATASET / subconjunto).glob("*.stem.mp4"))
    if limite:
        rutas = rutas[:limite]
    filas = ["mix_path,vocals_path,accompaniment_path,duration"]
    for ruta in rutas:
        nombre = ruta.name[: -len(".stem.mp4")].replace(",", "")
        carpeta = DIR_WAV / subconjunto / nombre
        carpeta.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            [
                "ffmpeg", "-nostdin", "-v", "error", "-y", "-i", str(ruta),
                "-map", "0:0", "-ar", "44100", "-ac", "2",
                str(carpeta / "mix.wav"),
                "-map", "0:4", "-ar", "44100", "-ac", "2",
                str(carpeta / "vocals.wav"),
                "-filter_complex",
                "[0:1][0:2][0:3]amix=inputs=3:normalize=0[a]",
                "-map", "[a]", "-ar", "44100", "-ac", "2",
                str(carpeta / "accompaniment.wav"),
            ],
            check=True,
        )
        tasa, datos = wavfile.read(carpeta / "mix.wav")
        base = f"{subconjunto}/{nombre}"
        filas.append(
            f"{base}/mix.wav,{base}/vocals.wav,{base}/accompaniment.wav,"
            f"{len(datos) / tasa:.3f}"
        )
    csv = TRABAJO / f"{subconjunto}.csv"
    csv.write_text("\n".join(filas) + "\n")
    print(f"{subconjunto}: {len(filas) - 1} canciones -> {csv.name}")


preparar("train", CONFIG["pistas_entrenamiento"])
preparar("test", CONFIG["pistas_prueba"])

## 4. Modelo preentrenado y métrica

Pesos oficiales de Spleeter 2stems (versión 1.4.0 de Deezer). Se usa el SDR
de la competencia Music Demixing (MDX), en dB; más alto es mejor.

In [ ]:
URL_MODELO = (
    "https://github.com/deezer/spleeter/releases/download/v1.4.0/"
    "2stems.tar.gz"
)
if not (DIR_PREENTRENADO / "2stems" / ".probe").exists():
    (DIR_PREENTRENADO / "2stems").mkdir(parents=True, exist_ok=True)
    archivo, _ = urllib.request.urlretrieve(URL_MODELO)
    with tarfile.open(archivo) as tar:
        tar.extractall(DIR_PREENTRENADO / "2stems")
    (DIR_PREENTRENADO / "2stems" / ".probe").write_text("OK")
print(sorted(p.name for p in (DIR_PREENTRENADO / "2stems").iterdir()))

CANCIONES_PRUEBA = sorted((DIR_WAV / "test").iterdir())
FUENTES = ["vocals", "accompaniment"]


def leer(ruta):
    _, datos = wavfile.read(ruta)
    return datos.astype(np.float64)


def sdr(referencia, estimacion, eps=1e-8):
    n = min(len(referencia), len(estimacion))
    referencia, estimacion = referencia[:n], estimacion[:n]
    num = (referencia**2).sum() + eps
    den = ((referencia - estimacion) ** 2).sum() + eps
    return 10 * np.log10(num / den)


def evaluar(parametros, salida, entorno=None):
    """Separa las canciones de prueba con el modelo y calcula el SDR."""
    spleeter(
        "separate", "-p", parametros, "-o", str(salida),
        "-f", "{foldername}/{instrument}.{codec}",
        *[str(c / "mix.wav") for c in CANCIONES_PRUEBA],
        entorno=entorno,
    )
    resultados = {f: [] for f in FUENTES}
    for cancion in CANCIONES_PRUEBA:
        for f in FUENTES:
            estimacion = leer(salida / cancion.name / f"{f}.wav")
            resultados[f].append(sdr(leer(cancion / f"{f}.wav"), estimacion))
    return {f: round(float(np.mean(v)), 3) for f, v in resultados.items()}


inicio = time.time()
sdr_antes = evaluar(
    "spleeter:2stems",
    TRABAJO / "separado_antes",
    {**ENTORNO, "MODEL_PATH": str(DIR_PREENTRENADO)},
)
print(f"SDR del modelo preentrenado ({time.time() - inicio:.0f} s):", sdr_antes)

## 5. Fine-tuning

La orden `spleeter train` solo continúa un checkpoint completo, y el oficial
no trae el estado del optimizador. Por eso se entrena con el mismo código de
Spleeter (`model_fn` y sus datasets), pero inicializando la red con los
pesos preentrenados (*warm start*).

In [ ]:
codigo = '''
import json
import sys
from functools import partial

import tensorflow as tf
from spleeter.audio.adapter import AudioAdapter
from spleeter.dataset import get_training_dataset, get_validation_dataset
from spleeter.model import model_fn
from spleeter.model.provider import ModelProvider

tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.INFO)
config, datos, preentrenado = sys.argv[1:4]
params = json.load(open(config))
calentamiento = tf.estimator.WarmStartSettings(
    ckpt_to_initialize_from=preentrenado,
    vars_to_warm_start=["^(?!.*(Adam|beta1_power|beta2_power|global_step)).*"],
)
estimador = tf.estimator.Estimator(
    model_fn=model_fn,
    model_dir=params["model_dir"],
    params=params,
    config=tf.estimator.RunConfig(
        save_checkpoints_steps=params["save_checkpoints_steps"],
        tf_random_seed=params["random_seed"],
        save_summary_steps=params["save_summary_steps"],
        log_step_count_steps=10,
        keep_checkpoint_max=2,
    ),
    warm_start_from=calentamiento,
)
adaptador = AudioAdapter.default()
tf.estimator.train_and_evaluate(
    estimador,
    tf.estimator.TrainSpec(
        input_fn=partial(get_training_dataset, params, adaptador, datos),
        max_steps=params["train_max_steps"],
    ),
    tf.estimator.EvalSpec(
        input_fn=partial(get_validation_dataset, params, adaptador, datos),
        steps=None,
        throttle_secs=params["throttle_secs"],
    ),
)
ModelProvider.writeProbe(params["model_dir"])
'''
(TRABAJO / "entrenar_spleeter.py").write_text(codigo)

# Configuración: la de 2stems con los cambios del fine-tuning
ruta_base = subprocess.run(
    [PY, "-c", "import spleeter.resources as r, os; "
     "print(os.path.join(os.path.dirname(r.__file__), '2stems.json'))"],
    capture_output=True, text=True, check=True,
).stdout.strip()
parametros = json.loads(Path(ruta_base).read_text())
parametros.update(
    train_csv=str(TRABAJO / "train.csv"),
    validation_csv=str(TRABAJO / "test.csv"),
    model_dir=str(DIR_SALIDA / "2stems"),
    T=CONFIG["T"],
    chunk_duration=CONFIG["duracion_fragmento"],
    batch_size=CONFIG["lote"],
    learning_rate=CONFIG["tasa_aprendizaje"],
    train_max_steps=CONFIG["pasos"],
    random_seed=CONFIG["semilla"],
    save_checkpoints_steps=max(CONFIG["pasos"] // 10, 1),
    throttle_secs=60,
    training_cache=str(TRABAJO / "cache" / "entrenamiento"),
    validation_cache=str(TRABAJO / "cache" / "validacion"),
)
CONFIG_JSON = TRABAJO / "2stems_finetuning.json"
CONFIG_JSON.write_text(json.dumps(parametros, indent=2))

In [ ]:
historial = []
inicio = time.time()
proceso = subprocess.Popen(
    [PY, str(TRABAJO / "entrenar_spleeter.py"), str(CONFIG_JSON),
     str(DIR_WAV), str(DIR_PREENTRENADO / "2stems")],
    env=ENTORNO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for linea in proceso.stdout:
    paso = re.search(r"loss = ([\d.]+), step = (\d+)", linea)
    if paso:
        historial.append((int(paso.group(2)), float(paso.group(1))))
        print(f"paso {paso.group(2):>6} | pérdida {paso.group(1)}")
    elif "Saving dict for global step" in linea:
        print("validación:", linea.split("Saving dict for global step")[1])
assert proceso.wait() == 0, "El entrenamiento falló"
print(f"Entrenamiento terminado en {time.time() - inicio:.0f} s")

## 6. Resultados

In [ ]:
sdr_despues = evaluar(str(CONFIG_JSON), TRABAJO / "separado_despues")
print(f"{'fuente':14} {'antes':>8} {'después':>8} {'mejora':>8}")
for f in FUENTES:
    mejora = sdr_despues[f] - sdr_antes[f]
    print(f"{f:14} {sdr_antes[f]:8.3f} {sdr_despues[f]:8.3f} {mejora:+8.3f}")

if historial:
    pasos, perdidas = zip(*historial)
    plt.figure(figsize=(7, 4))
    plt.plot(pasos, perdidas)
    plt.xlabel("Paso")
    plt.ylabel("Pérdida")
    plt.title("Fine-tuning de Spleeter 2stems con MUSDB18")
    plt.grid(alpha=0.3)
    plt.savefig(DIR_SALIDA / "perdida.png", dpi=120, bbox_inches="tight")
    plt.show()

(DIR_SALIDA / "metricas.json").write_text(
    json.dumps(
        {
            "configuracion": CONFIG,
            "sdr_antes": sdr_antes,
            "sdr_despues": sdr_despues,
            "perdida_por_paso": historial,
        },
        indent=2,
        ensure_ascii=False,
    )
)
(DIR_SALIDA / "2stems_finetuning.json").write_text(CONFIG_JSON.read_text())
print(sorted(p.name for p in DIR_SALIDA.rglob("*") if p.is_file()))

In [ ]:
# Descarga los resultados (solo en Google Colab)
try:
    from google.colab import files

    subprocess.run(
        ["zip", "-qr", "modelo_afinado.zip", DIR_SALIDA.name], cwd=TRABAJO
    )
    files.download(str(TRABAJO / "modelo_afinado.zip"))
except ImportError:
    print("Resultados en", DIR_SALIDA)